In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.getcwd()))

In [5]:
from helpers import load_config
print(load_config.__module__)

config_loader


In [2]:
from markov_models import MarkovModel 
from info_rate import compute_info_rate
from helpers import update_values_in_csv, check_data_availability
from helpers import load_config
import numpy as np
import pickle


def main():
    languages = ['ENG'] # ['ENG', 'FRA', 'DEU', "ENG",'JPN', 'VIE', 'YUE']

    for language in languages:
        config_dict = load_config(language)

        for processing_type in ['phones','sylls']: 
            print(f"\nLanguage: {language}")
            print(f"Processing type: {processing_type.upper()}")
            print(f"=======================================================================================")

            input_path = check_data_availability(language, processing_type, config_dict)
            if input_path: 
                with open(input_path, "rb") as f:
                    data = pickle.load(f)
                    #data = data[:100]
                    print(data[:5])
                
            else: continue
            
            for text_type in ['words', 'sentences']:
                print(f"\n📊 Computing ID and IR across {text_type.upper()}")

                n_values = [1, 2, 3, 4]  # For bigram, trigram, and quadgram models
                markov_models = {}

                for n in n_values:

                    print(f"\n🧮 Training a Markov Model with n = {n}:")

                    # Create and build the Markov model
                    model = MarkovModel(n)

                    # Build the markov model
                    model.build(data, text_type)

                    # Compute the conditional entropy (information density)
                    info_density = model.compute_conditional_entropy()
                    print(f"Information Density: {info_density:.4f}")

                    # Compute the information rate (bits per second)
                    info_rate_values, speech_rate_values = compute_info_rate(info_density, processing_type, language)
                    print(f"Information Rate: {np.mean(info_rate_values):.4f}")
                    
                    # Update the CSV file with the computed values
                    update_values_in_csv(language, info_density, n, 'ID', text_type, processing_type)
                    update_values_in_csv(language, info_rate_values, n, 'IR', text_type, processing_type)
                    update_values_in_csv(language, speech_rate_values, n, 'SR', text_type, processing_type)
                    

                    # Store model for later use 
                    markov_models[n] = model

                    # Display exactly 3 examples
                    """example_count = 0
                    print("\nExample probabilities (p(x, y)):")

                    for (prefix, suffix), p_xy in model.cond_probs.items():
                        print(f"p({prefix} -> {suffix}) = {p_xy:.4f}")
                        example_count += 1
                        # if example_count == 3:
                            break"""
                    
                    # Save the model to a file
                    model.save_model(language, processing_type, text_type)

        # For plotting, see plotting.ipynb


if __name__ == "__main__":
    import cProfile
    import pstats

    profiler = cProfile.Profile()
    profiler.enable()

    main()

    profiler.disable()
    stats = pstats.Stats(profiler).sort_stats('cumulative')
    stats.print_stats(30)  # Top 30 calls by cumulative time

Exception ignored When destroying _lsprof profiler:
Traceback (most recent call last):
  File "/tmp/ipykernel_1965859/224026787.py", line 83, in <module>
RuntimeError: Cannot install a profile function while another profile function is being installed


TypeError: load_config() missing 1 required positional argument: 'key'